CAP4770 Final Project
Data source
https://www.datasciencedata.com/auto-insurance-churn-data
Created by: Matt Strenges, Tuyet Phan, and Mary Szabo
Summer 2023

In [ ]:
import pylab
import matplotlib
import pandas as pd
import numpy as np
import os
import pandas as pd
from sklearn.tree import DecisionTreeClassifier # Import Decision Tree Classifier
from sklearn.model_selection import train_test_split,RandomizedSearchCV, train_test_split # Import train_test_split function
from sklearn import metrics #Import scikit-learn metrics module for accuracy calculation
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from scipy.stats import randint
from IPython.display import Image

df = pd.read_csv("full_auto_insurance_data.csv")

In [ ]:
display(df) #Original DF

In [ ]:
#Processing Data
df.loc[df['CURR_ANN_AMT'].isnull(), 'CURR_ANN_AMT'] = 931.06 #mean CURR_ANN_AMT is 931
df.loc[df['DAYS_TENURE'].isnull(), 'DAYS_TENURE'] = 3482.62 # mean DAYS_TENURE is 3482
df.loc[df['AGE_IN_YEARS'].isnull(), 'AGE_IN_YEARS'] = 55.44 # mean AGE_IN_YEARS is 55.44
df.loc[df['INCOME'].isnull(), 'INCOME'] = 80372.36 # mean INCOME is 80372.36
df.loc[df['HAS_CHILDREN'].isnull(), 'HAS_CHILDREN'] = 0.0 # mode HAS_CHILDREN is 0
df.loc[df['LENGTH_OF_RESIDENCE'].isnull(), 'LENGTH_OF_RESIDENCE'] = 6.80 # mean LENGTH_OF_RESIDENCE is 6.80
df.loc[df['MARITAL_STATUS'].isnull(), 'MARITAL_STATUS'] = "Married" # mode MARITAL_STATUS is Married
df.loc[df['MARITAL_STATUS']=="Married", 'MARITAL_STATUS'] = 1 #Married is a 1
df.loc[df['MARITAL_STATUS']=="Single", 'MARITAL_STATUS'] = 0 #Single is a 0
df.loc[df['HOME_MARKET_VALUE'].isnull(), 'HOME_MARKET_VALUE'] = "75000 - 99999" # mode HOME_MARKET_VALUE is 75000 - 99999
df.loc[df['HOME_OWNER'].isnull(), 'HOME_OWNER'] = 1 # mode HOME_OWNER is 1
df.loc[df['COLLEGE_DEGREE'].isnull(), 'COLLEGE_DEGREE'] = 0 # mode COLLEGE_DEGREE is 0
df.loc[df['GOOD_CREDIT'].isnull(), 'GOOD_CREDIT'] = 1 # mode GOOD_CREDIT is 1
df.loc[df['ACCT_SUSPD_DATE'].notnull(), 'acct_terminated'] = 1 #terminated=1
df.loc[df['ACCT_SUSPD_DATE'].isnull(), 'acct_terminated'] = 0 #not terminated=0
df = df.drop('ACCT_SUSPD_DATE', axis=1)
df = df.drop('DATE_OF_BIRTH', axis=1)
df = df.drop('SOCIAL_SECURITY_NUMBER', axis=1)
df = df.drop('LATITUDE', axis=1)
df = df.drop('LONGITUDE', axis=1)
df = df.drop('STREET_ADDRESS', axis=1)
df = df.drop('CITY', axis=1)
df = df.drop('STATE', axis=1)
df = df.drop('COUNTY', axis=1)
df = df.dropna(axis=0)
pd.set_option('display.max_columns', None) #Displays every column
df #Processed DF

#These data columns were not changed as I don't believe we are using them and changing them would be complicated.
#ADDRESS_ID was not changed
#CUST_ORIG_DATE was not changed
#DATE_OF_BIRTH was not changed
#SOCIAL_SECURITY_NUMBER was not changed
#LATITUDE & LONGITUDE were not changed
#STREET, CITY, STATE & COUNTY were not changed

In [ ]:
#col_names = [ 'CURR_ANN_AMT', 'DAYS_TENURE', 'AGE_IN_YEARS', 'INCOME', 'HAS_CHILDREN', 'LENGTH_OF_RESIDENCE',   'HOME_OWNER', 'COLLEGE_DEGREE', 'GOOD_CREDIT']
cols = ['CURR_ANN_AMT', 'DAYS_TENURE', 'AGE_IN_YEARS', 'INCOME', 'HAS_CHILDREN', 'LENGTH_OF_RESIDENCE',  'HOME_OWNER', 'COLLEGE_DEGREE', 'GOOD_CREDIT']
X = df[cols]
y = df.acct_terminated
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

clf = DecisionTreeClassifier()
clf = clf.fit(X_train,y_train)
y_pred = clf.predict(X_test)
print("Accuracy:",metrics.accuracy_score(y_test, y_pred))

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

In [ ]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

In [ ]:
y_pred2 = rf.predict(X_test)
accuracy2 = accuracy_score(y_test, y_pred2)
print("Random Forest Accuracy:", accuracy2)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred2))

Oversampling

In [ ]:
# https://machinelearningmastery.com/random-oversampling-and-undersampling-for-imbalanced-classification/

from imblearn.over_sampling import RandomOverSampler
from collections import Counter
# from sklearn.model_selection import train_test_split

# Separate the independent variables from the dependent variable
X_over = df.drop('acct_terminated', axis=1)  # Features (independent variables)
y_over = df['acct_terminated']  # Target variable (dependent variable)

# Create the RandomOverSampler instance targeting the minority class
oversampler = RandomOverSampler(sampling_strategy='minority')

# Apply oversampling to the dataset
X_resampled, y_resampled = oversampler.fit_resample(X_over, y_over)

print("Before oversampling: ", Counter(y_over))
print("After oversampling: ", Counter(y_resampled))

In [ ]:
# Convert the resampled arrays back to DataFrames
X_resampled_df = pd.DataFrame(X_resampled, columns=X_over.columns)
y_resampled_df = pd.DataFrame(y_resampled, columns=['acct_terminated'])

resampled_df = pd.concat([X_resampled_df, y_resampled_df], axis=1)

print(resampled_df)

In [ ]:
resampled_df.to_csv('oversampling_data.csv', index=False)

In [ ]:
df_over = pd.read_csv('oversampling_data.csv')

In [ ]:
cols = ['CURR_ANN_AMT', 'DAYS_TENURE', 'AGE_IN_YEARS', 'INCOME', 'HAS_CHILDREN', 'LENGTH_OF_RESIDENCE',  'HOME_OWNER', 'COLLEGE_DEGREE', 'GOOD_CREDIT']
X_over = df[cols]
y_over = df.acct_terminated
X_train, X_test, y_train, y_test = train_test_split(X_over, y_over, test_size=0.2, random_state=1)

# Create the DecisionTreeClassifier instance
model = DecisionTreeClassifier()

# Fit the model on the training data
model.fit(X_train, y_train)

# Predict on the test data
preds = model.predict(X_test)

accuracy = accuracy_score(y_test, preds)
print('Decision Tree accuracy:', accuracy)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, preds))

In [ ]:
# Randomly sample 10% of the data for faster testing
sampled_df = df.sample(frac=0.1, random_state=1)

cols = ['CURR_ANN_AMT', 'DAYS_TENURE', 'AGE_IN_YEARS', 'INCOME', 'HAS_CHILDREN', 'LENGTH_OF_RESIDENCE', 'HOME_OWNER', 'COLLEGE_DEGREE', 'GOOD_CREDIT']
X_over = sampled_df[cols]
y_over = sampled_df['acct_terminated']

# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X_over, y_over, test_size=0.2, random_state=1)

# Create the RandomForestClassifier instance with n_jobs=-1 for parallel processing
model = RandomForestClassifier(n_estimators=100, n_jobs=-1)

# Fit the model on the training data
model.fit(X_train, y_train)

# Predict on the test data
preds = model.predict(X_test)

accuracy = accuracy_score(y_test, preds)
print('Random Forest accuracy:', accuracy)

In [ ]:
print(classification_report(y_test, preds))

Undersampling

In [ ]:
#Before I undersample, I want to see the starting values for terminated vs active accounts 
#There table terminated accounts 269,259 initially before rows with NaN were deleted. 
terminatedCount = len(df[df["acct_terminated"]==1])
print(terminatedCount)
activeCount = len(df[df["acct_terminated"]==0])
print(activeCount)

In [ ]:
#For undersampling, the number of tuples of terminated and active accounts should be the same
#After removing the NaN rows, we are only left with 27,542 terminated accounts and 220220 active accounts
#Next, we need the number of active accouts to be equal to the number of terminated accounts. 
active = df[df['acct_terminated'] == 0]
active = active.head(269259) #take the first 269259 rows
terminated = df[df['acct_terminated'] == 1]

In [ ]:
#Combine the active and terminated dataframes
frames = [active, terminated]
df_undersample = pd.concat(frames)
df_undersample

In [ ]:
#col_names = [ 'CURR_ANN_AMT', 'DAYS_TENURE', 'AGE_IN_YEARS', 'INCOME', 'HAS_CHILDREN', 'LENGTH_OF_RESIDENCE',   'HOME_OWNER', 'COLLEGE_DEGREE', 'GOOD_CREDIT']
cols = ['CURR_ANN_AMT', 'DAYS_TENURE', 'AGE_IN_YEARS', 'INCOME', 'HAS_CHILDREN', 'LENGTH_OF_RESIDENCE',  'HOME_OWNER', 'COLLEGE_DEGREE', 'GOOD_CREDIT']
X = df_undersample[cols]
y = df_undersample.acct_terminated
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=5)

clf = DecisionTreeClassifier()
clf = clf.fit(X_train,y_train)
y_pred = clf.predict(X_test)
print("Accuracy:",metrics.accuracy_score(y_test, y_pred))

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

In [ ]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

In [ ]:
y_pred2 = rf.predict(X_test)
accuracy2 = accuracy_score(y_test, y_pred2)
print("Random Forest Accuracy:", accuracy2)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred2))